# PART 2: How to classify a RGB drone dataset into vegetation types

## Introduction

Executing the code in this notebook allows to classify land covers within a given study area using UAV RGB data and Canopy Height Model (CHM) information. The code runs as-is with the provided dummy example. Before running the code make sure to have done all the preparatory steps mentioned in Part 1 of this tutorial. 

To run this notebook you need to have used the file *classification.yml* to create and activate a python environment that has all the necessary packages installed, and you need to have downloaded the auxiliary code (*classification_aux.py*) as well as the *dummy_example* folder and the classifier library (*classifier_lib*). See Part 1 of this tutorial if you haven't.

Let's get started by loading all the necessary packages in **Cell 1**:

In [1]:
# CELL 1

import pickle as pkl
import os
import rasterio as rio
import numpy as np
from pathlib import Path
from classification_aux import classify_mission, load_site_data, feature_mission

## Load data
In **Cell 2 and 3**, the raw RGB and CHM datasets are loaded and split into chunks. The datasets are split to enable procesing of larger datasets on computers with limited processing power. The number of chunks into which the datasets are splitted is indicated by the variable *chunks* in Cell 3. Note, the amount of chunks created by the program results from multiplying *chunks* by itself *chunks* times, so if *chunks* = 3 the real number of splits = 3*3 = 9. We set the default *chunks* value to three, but if your computer runs into memory problems you can increase the number and set *chunks* = 6 for example. 

After splitting, features are calculated for each pixel in the RGB dataset by iterating through the chunks. Features are the y-variables the classification models use as input to calculate a land cover class. The features are, for example, some indices calculated from the RGB values or chromaticity. For an exhaustive list of all features take a look at the function *feature_mission()* in the *classification_aux.py* file or read our publication.

**Cell 2** defines the input directory (*data_dir*) where the RGB and CHM raw data are loaded from. It also defines the output directory (*run_dir*) where the chunks are saved. 

In **Cell 3** the function *load_site_data()* is used to load the RGB and CHM data from the dummy datasets. The output from *load_site_data()* is then passed to the function *feature_mission()* to calculate all necessary features for classification.

In [2]:
# Cell 2 - define data directories

# location of raw data
data_dir = os.path.join('dummy_data/siteXY')                              # this is a relative path, it is therefore important that there is no leading '/'
                                                                          # and that the dummy_data folder is saved in the same folder as this notebook
# place where chunked feature data is stored
site = 'siteXY'
run_name = f'{site}_01'                                                   # this is used to name the chunks that are saved in run_dir
run_dir = os.path.join('dummy_data', 'classifier_runs', run_name)

if not os.path.exists(run_dir):                                           # creates the path to save chunks in case it doesn't exist
    os.makedirs(os.path.join(run_dir, 'composite'))

In [3]:
# Cell 3 - load data and calculate features

fname_rgb = 'siteXY_orthomosaic'                                # name of the rgb file
fname_chm = 'siteXY_chm_stratified'                             # name of the chm file
rasters = load_site_data(data_dir, fname_rgb, fname_chm)        # load site data

chunks = 3                                                      # determine rows and cols the dataset is split into
feature_mission(rasters, chunks, run_dir)                       # calculate features by iterating through chunks

looking for rgb.tif file here: C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/code/generic_git/documentation/dummy_data/siteXY/siteXY_orthomosaic.tif
looking for chm.tif file here: C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/code/generic_git/documentation/dummy_data/siteXY/dem_and_derivatives/siteXY_chm_stratified.tif
Writing chunk 0-0
Writing chunk 0-1
Writing chunk 0-2
Writing chunk 1-0
Writing chunk 1-1
Writing chunk 1-2
Writing chunk 2-0
Writing chunk 2-1
Writing chunk 2-2


## Loading the classifier 

In **Cell 4**, the classifier model is loaded from a .pkl file stored in the classifier library. There are three choices of classifiers:
1. One generic classifier that was trained on data from 42 UAV missions spread across two ecozones (*RF_generic.pkl*)
2. One generic classifier trained on data from 18 UAV missions spread across the Taiga Shield.     (*RF_generic_Tshield.pkl*)
3. One generic classifier trained on data from 23 UAV missions spread across the Taiga Plains.     (*RF_generic_Tplains.pkl*)

Note, you can choose one of the three classifiers by deleting the '#' in front of the corresponding clf_name and by adding a '#' in front of the two other clf_names. By default the overall generic classifier is activated (*RF_generic.pkl*). 

In [4]:
# Cell 4 - load the classifier model

#clf_dir = 'C:/Users/MABEB16/science/postdoc/5_projects/1_general_classifier/4_general_classifier/results/2a_classifier/RF_generic/'
clf_dir = 'dummy_data/classifier_lib/generic_RF/
clf_name = f'RF_generic.pkl'            # generic classifier trained on UAV data from two ecozones (see also publication)
#clf_name = f'RF_generic_Tshield.pkl'   # generic classifier trained on UAV data from the Taiga Shield
#clf_name = f'RF_generic_Tplains.pkl'   # generic classifier trained on UAV data from the Taiga Plains


with open (clf_dir+clf_name, 'rb') as file:
    model = pkl.load(file)
#with open(clf_dir+f'RF_{site}.pkl', 'rb') as file:      
#    model = pkl.load(file) 

## Classify drone data

In **Cell 5** the classifier chosen in Cell 4 is used to determine land cover classes for the study area by iterating through the chunks produced in Cell 3. For more information see the *classify_mission()* function in the *classification_aux.py* file.

In [5]:
# Cell 5 - classify drone data

# define variables that are passed to the classify_mission() function
blocks = chunks
rgb_file = f'{data_dir}/{fname_rgb}'
run_dir = f'dummy_data/classifier_runs/{run_name}'

# run the classify_mission function
output = classify_mission(blocks=blocks, rgb_file=rgb_file, run_dir=run_dir, model=model)

C:\Users\MABEB16\science\postdoc\5_projects\1_general_classifier\4_general_classifier\code\generic_git\documentation
Processing block 0-0
Processing block 0-1
Processing block 0-2
Processing block 1-0
Processing block 1-1
Processing block 1-2
Processing block 2-0
Processing block 2-1
Processing block 2-2


## Save result as .tiff

**Cell 6** is the final step where the classified chunks are stiched together again to form one .tif file. Properties of this new .tif (like projection) are copied from the rgb.tif. 

The directory to store the classified file and the file's name are defined in the *out_dir* and *out_file* variables. By default, the out_dir variables points to a *results* folder within the *dummy_example* folder. If the folder doesn't exist it will be created. The default file name is *dummy_classified_RFall.tif*.

The results stored in the reulsts .tif file are numbers representing land cover classes:
 1. Cladonia lichen
 2. 

In [6]:
# Cell 6 saving classification results to .tif

out_dir = f'dummy_data/result/'                              # path to save the results -> change if you want to store results elsewhere
os.makedirs(out_dir, exist_ok=True)                          # creates 'results' folder if doesn't exist
out_file = 'dummy_classified_RFall.tif'                      # filename of results .tif -> change if you want different name

rgb_file = f'{data_dir}/{fname_rgb}.tif'                     # load rgb file and copy .tif properties like projection etc. into the 'meta' variable
with rio.open(rgb_file) as f:
    meta = f.meta
meta.update(count=1)

with rio.open(out_dir + out_file, 'w', **meta) as dst:       # Write a new .tif file using the metadata from the original file
    dst.write(output, 1)